# nb_21_silver_sales

Silver, part 2. Starts from the cleaned tables of part 1 and produces the **sales table** the rest of the project uses.

What happens
1. **Reference tables** are typed: `silver_product`, `silver_boutique`, and `silver_fx_rate_daily`. The daily exchange rate table has a rate for **every day**. Weekends and holidays carry the last known rate forward.
2. **Duplicates** that the source sent twice are removed. The first copy is kept. Each removed copy is recorded in `ctl_quarantine` with the rule `resent_duplicate`.
3. **Boutique and online sales are combined** into `silver_sales`, with a `channel` column, product category, and every amount in local currency and in EUR.
4. **Returns are linked** to the sale they undo (same client, same product, same channel, within 14 days).
5. **Repairs** are de-duplicated into `silver_after_sales`.
6. Reconciliation proves that nothing was lost: `rows in = rows kept + duplicates removed + rows rejected`.

Input: `silver_pos_transaction`, `silver_ecom_order`, `silver_after_sales_ticket` (from nb_20), `bronze_product`, `bronze_boutique`, `bronze_fx_rate`

Output: `silver_sales`, `silver_after_sales`, `silver_product`, `silver_boutique`, `silver_fx_rate_daily`, plus rows in `ctl_quarantine`, `ctl_dq_results`, `ctl_run_log`

In [ ]:
# ---- Settings -------------------------------------------------------------------------
# Return window in days. Source: the CHANEL US returns FAQ says 14 days from purchase.
RETURN_WINDOW_DAYS = 14

In [ ]:
# ---- Setup ----------------------------------------------------------------------------
import uuid
from datetime import datetime, timezone
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import Row, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType,
                               DoubleType, TimestampType)

SILVER_BATCH_ID = "S" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print("silver batch id:", SILVER_BATCH_ID)


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


dq_rows, run_rows, quarantine_dfs, gaps = [], [], [], []


def add_dq(table, rule, severity, checked, failed):
    dq_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", table_name=table, rule=rule, severity=severity,
                       rows_checked=int(checked), rows_failed=int(failed),
                       pct_failed=round(100.0 * failed / checked, 3) if checked else 0.0, checked_at=now_utc()))


def write_table(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    return spark.table(name).count()


def log_run(obj, status, rows, started, message, failure_class=None):
    run_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", object_name=obj, status=status, files_loaded=0,
                        rows_loaded=int(rows), started_at=started, finished_at=now_utc(),
                        failure_class=failure_class, message=message))


def quarantine_rows(df, stage, source_table, key_col, rule, failure_class, raw_cols):
    return df.select(
        F.lit(SILVER_BATCH_ID).alias("batch_id"), F.lit(stage).alias("stage"), F.lit(source_table).alias("source_table"),
        F.col("source_file"), F.col(key_col).alias("record_key"), F.lit(rule).alias("rule"), F.lit(rule).alias("all_rules"),
        F.lit(failure_class).alias("failure_class"), F.to_json(F.struct(*raw_cols)).alias("raw_record"),
        F.current_timestamp().alias("quarantined_at"))

In [ ]:
# ---- Reference tables -----------------------------------------------------------------
started = now_utc()
product = (spark.table("bronze_product")
           .select("product_id", "category", "sub_category", "name",
                   F.col("list_price_eur").cast("decimal(18,2)").alias("list_price_eur")))
n = write_table(product, "silver_product")
log_run("silver_product", "success", n, started, f"rows={n}")

started = now_utc()
boutique = spark.table("bronze_boutique").select("boutique_id", "city", "country", "region", "currency")
n = write_table(boutique, "silver_boutique")
log_run("silver_boutique", "success", n, started, f"rows={n}")
print("silver_product, silver_boutique written")

In [ ]:
# ---- Exchange rates: one rate for every day -------------------------------------------
# The source publishes rates on weekdays only. A weekend sale uses the last published rate.
started = now_utc()
fx_raw = (spark.table("bronze_fx_rate")
          .select(F.to_date("rate_date").alias("rate_date"), "currency", F.col("rate_to_eur").cast("decimal(18,8)").alias("rate_to_eur")))
fx_bad = fx_raw.filter(F.col("rate_date").isNull() | F.col("rate_to_eur").isNull() | (F.col("rate_to_eur") <= 0)).count()
add_dq("bronze_fx_rate", "fx_rate_invalid", "reject", fx_raw.count(), fx_bad)
fx_ok = fx_raw.filter(F.col("rate_date").isNotNull() & (F.col("rate_to_eur") > 0))

first_day = fx_ok.agg(F.min("rate_date")).collect()[0][0]
last_sale_day = max(
    spark.table("silver_pos_transaction").agg(F.max("event_date")).collect()[0][0],
    spark.table("silver_ecom_order").agg(F.max("event_date")).collect()[0][0])
last_fx_day = fx_ok.agg(F.max("rate_date")).collect()[0][0]
last_day = max(last_sale_day, last_fx_day)

spine = (spark.range(1).select(F.explode(F.sequence(F.lit(first_day), F.lit(last_day), F.expr("interval 1 day"))).alias("rate_date"))
         .crossJoin(fx_ok.select("currency").distinct()))
w = Window.partitionBy("currency").orderBy("rate_date").rowsBetween(Window.unboundedPreceding, 0)
fx_daily = (spine.join(fx_ok, ["rate_date", "currency"], "left")
            .withColumn("rate_source_date", F.last(F.when(F.col("rate_to_eur").isNotNull(), F.col("rate_date")), ignorenulls=True).over(w))
            .withColumn("rate_to_eur", F.last("rate_to_eur", ignorenulls=True).over(w))
            .withColumn("is_carried_forward", F.col("rate_date") != F.col("rate_source_date"))
            .select("rate_date", "currency", "rate_to_eur", "rate_source_date", "is_carried_forward"))
n = write_table(fx_daily, "silver_fx_rate_daily")
carried = spark.table("silver_fx_rate_daily").filter("is_carried_forward").count()
add_dq("silver_fx_rate_daily", "rate_carried_forward", "info", n, carried)
log_run("silver_fx_rate_daily", "success", n, started, f"rows={n} carried_forward={carried}")
print(f"silver_fx_rate_daily: {n} rows, {carried} carried forward from the previous day")

In [ ]:
# ---- Remove duplicates ----------------------------------------------------------------
# A source sometimes sends yesterday's row again. The first copy is kept.
# If two copies of one key have different content, that is counted as a conflict (the first copy still wins).
def dedupe(table, key_col):
    df = spark.table(table)
    lineage = {"source_file", "file_date", "bronze_batch_id", "silver_batch_id"}
    content_cols = [c for c in df.columns if c not in lineage]
    df = df.withColumn("_content", F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in content_cols]), 256))
    w = Window.partitionBy(key_col).orderBy(F.col("file_date").asc(), F.col("source_file").asc())
    df = (df.withColumn("_rn", F.row_number().over(w))
            .withColumn("_distinct_content", F.size(F.collect_set("_content").over(Window.partitionBy(key_col)))))
    df = df.cache()
    total = df.count()
    dups = df.filter("_rn > 1")
    dup_count = dups.count()
    conflicts = df.filter("_rn = 1 and _distinct_content > 1").count()
    add_dq(table, "resent_duplicate", "fix", total, dup_count)
    add_dq(table, "conflicting_duplicate_keys", "info", total, conflicts)
    quarantine_dfs.append(quarantine_rows(dups, "silver_dedupe", table, key_col, "resent_duplicate", "source", content_cols))
    kept = df.filter("_rn = 1").drop("_content", "_rn", "_distinct_content")
    return kept, total, dup_count, conflicts


pos_kept, pos_total, pos_dups, pos_conf = dedupe("silver_pos_transaction", "transaction_id")
ecom_kept, ecom_total, ecom_dups, ecom_conf = dedupe("silver_ecom_order", "order_id")
tk_kept, tk_total, tk_dups, tk_conf = dedupe("silver_after_sales_ticket", "ticket_id")
print(f"duplicates removed  boutique sales: {pos_dups}   online orders: {ecom_dups}   repairs: {tk_dups}")
print(f"conflicting copies  boutique sales: {pos_conf}   online orders: {ecom_conf}   repairs: {tk_conf}")

In [ ]:
# ---- One sales table, with category and EUR amounts -----------------------------------
started = now_utc()
pos_u = pos_kept.select(
    F.col("transaction_id").alias("sale_id"), F.lit("boutique").alias("channel"), "client_id", "boutique_id", "product_id",
    "quantity", "unit_price", "currency", "is_return", F.col("transaction_ts").alias("event_ts"), "event_date", "region",
    (F.col("sales_associate_id") if "sales_associate_id" in pos_kept.columns else F.lit(None).cast("string")).alias("sales_associate_id"),
    "source_file", "file_date", "bronze_batch_id")
ecom_u = ecom_kept.select(
    F.col("order_id").alias("sale_id"), F.lit("online").alias("channel"), "client_id",
    F.lit(None).cast("string").alias("boutique_id"), "product_id",
    "quantity", "unit_price", "currency", "is_return", F.col("order_ts").alias("event_ts"), "event_date", "region",
    F.lit(None).cast("string").alias("sales_associate_id"),
    "source_file", "file_date", "bronze_batch_id")
sales = pos_u.unionByName(ecom_u)

fx = spark.table("silver_fx_rate_daily").select(
    F.col("rate_date").alias("event_date"), "currency", F.col("rate_to_eur").alias("fx_rate_to_eur"),
    F.col("rate_source_date").alias("fx_rate_date"), F.col("is_carried_forward").alias("fx_carried_forward"))
prod = spark.table("silver_product").select("product_id", "category", "sub_category")

sales = sales.join(fx, ["event_date", "currency"], "left").join(prod, "product_id", "left").cache()

no_fx = sales.filter(F.col("fx_rate_to_eur").isNull())
no_fx_count = no_fx.count()
sales_in = pos_kept.count() + ecom_kept.count()
add_dq("silver_sales", "no_fx_rate", "reject", sales_in, no_fx_count)
quarantine_dfs.append(quarantine_rows(no_fx, "silver_sales", "silver_sales", "sale_id", "no_fx_rate", "integration",
                                      ["sale_id", "channel", "client_id", "product_id", "currency", "event_ts"]))
sales = sales.filter(F.col("fx_rate_to_eur").isNotNull())

sales = (sales
         .withColumn("line_amount_local", (F.col("quantity") * F.col("unit_price")).cast("decimal(18,2)"))
         .withColumn("unit_price_eur", (F.col("unit_price") * F.col("fx_rate_to_eur")).cast("decimal(18,4)"))
         .withColumn("line_amount_eur", F.round(F.col("quantity") * F.col("unit_price") * F.col("fx_rate_to_eur"), 2).cast("decimal(18,2)")))

In [ ]:
# ---- Link each return to the sale it undoes -------------------------------------------
# Same client, same product, same channel, bought no more than RETURN_WINDOW_DAYS before, and at least as many pieces.
# If several sales qualify, the most recent one wins.
returns = (sales.filter("is_return")
           .select(F.col("sale_id").alias("return_id"), "client_id", "product_id", "channel",
                   F.abs(F.col("quantity")).alias("returned_qty"), F.col("event_ts").alias("return_ts")))
originals = (sales.filter(~F.col("is_return"))
             .select(F.col("sale_id").alias("original_sale_id"), "client_id", "product_id", "channel",
                     F.col("quantity").alias("original_qty"), F.col("event_ts").alias("original_ts")))
cand = (returns.join(originals, ["client_id", "product_id", "channel"])
        .filter((F.col("original_ts") <= F.col("return_ts"))
                & (F.datediff(F.to_date("return_ts"), F.to_date("original_ts")) <= RETURN_WINDOW_DAYS)
                & (F.col("original_qty") >= F.col("returned_qty"))))
best = (cand.withColumn("_rn", F.row_number().over(Window.partitionBy("return_id").orderBy(F.col("original_ts").desc())))
        .filter("_rn = 1")
        .select(F.col("return_id").alias("sale_id"), "original_sale_id",
                F.datediff(F.to_date("return_ts"), F.to_date("original_ts")).alias("days_to_return")))

sales = sales.join(best, "sale_id", "left")

silver_sales = sales.select(
    "sale_id", "channel", "client_id", "boutique_id", "sales_associate_id", "product_id", "category", "sub_category",
    "quantity", "unit_price", "currency", "line_amount_local", "fx_rate_to_eur", "fx_rate_date", "fx_carried_forward",
    "unit_price_eur", "line_amount_eur", "is_return", "original_sale_id", "days_to_return",
    "event_ts", "event_date", "region", "source_file", "file_date", "bronze_batch_id",
    F.lit(SILVER_BATCH_ID).alias("silver_batch_id"))
sales_kept = write_table(silver_sales, "silver_sales")

n_returns = returns.count()
unmatched = n_returns - best.count()
add_dq("silver_sales", "return_without_original_sale", "info", n_returns, unmatched)
add_dq("silver_sales", "amount_converted_with_carried_forward_rate", "info", sales_kept,
       spark.table("silver_sales").filter("fx_carried_forward").count())

# Every row that came in must be kept, removed as a duplicate, or rejected
gap = (pos_total + ecom_total) - (sales_kept + pos_dups + ecom_dups + no_fx_count)
add_dq("silver_sales", "reconcile_cleaned_to_sales", "reconcile", pos_total + ecom_total, abs(gap))
log_run("silver_sales", "success" if gap == 0 else "failed", sales_kept, started,
        f"in={pos_total + ecom_total} kept={sales_kept} duplicates={pos_dups + ecom_dups} no_fx={no_fx_count} gap={gap}",
        None if gap == 0 else "processing")
if gap:
    gaps.append("silver_sales")
print(f"silver_sales  in={pos_total + ecom_total}  kept={sales_kept}  duplicates={pos_dups + ecom_dups}  no_fx_rate={no_fx_count}  gap={gap}")
print(f"returns: {n_returns}, linked to a sale: {n_returns - unmatched}, not linked: {unmatched}")

In [ ]:
# ---- Repairs and services -------------------------------------------------------------
started = now_utc()
tk_final = tk_kept.select("ticket_id", "client_id", "boutique_id", "product_id", "status", "reason", "created_at", "closed_at",
                          "purchase_region", "service_region", "is_cross_region_service",
                          (F.datediff(F.to_date("closed_at"), F.to_date("created_at"))).alias("days_to_close"),
                          "source_file", "file_date", "bronze_batch_id", F.lit(SILVER_BATCH_ID).alias("silver_batch_id"))
tk_kept_n = write_table(tk_final, "silver_after_sales")
tk_gap = tk_total - (tk_kept_n + tk_dups)
add_dq("silver_after_sales", "reconcile_cleaned_to_after_sales", "reconcile", tk_total, abs(tk_gap))
log_run("silver_after_sales", "success" if tk_gap == 0 else "failed", tk_kept_n, started,
        f"in={tk_total} kept={tk_kept_n} duplicates={tk_dups} gap={tk_gap}", None if tk_gap == 0 else "processing")
if tk_gap:
    gaps.append("silver_after_sales")
print(f"silver_after_sales  in={tk_total}  kept={tk_kept_n}  duplicates={tk_dups}  gap={tk_gap}")

In [ ]:
# ---- Write the control tables ---------------------------------------------------------
QUARANTINE_COLS = ["batch_id", "stage", "source_table", "source_file", "record_key", "rule",
                   "all_rules", "failure_class", "raw_record", "quarantined_at"]
q = reduce(lambda a, b: a.unionByName(b), quarantine_dfs).select(*QUARANTINE_COLS)
q.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("ctl_quarantine")

DQ_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("table_name", StringType()), StructField("rule", StringType()),
    StructField("severity", StringType()), StructField("rows_checked", LongType()),
    StructField("rows_failed", LongType()), StructField("pct_failed", DoubleType()),
    StructField("checked_at", TimestampType())])
RUN_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("object_name", StringType()), StructField("status", StringType()),
    StructField("files_loaded", IntegerType()), StructField("rows_loaded", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("failure_class", StringType()), StructField("message", StringType())])
spark.createDataFrame(dq_rows, schema=DQ_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_dq_results")
spark.createDataFrame(run_rows, schema=RUN_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_run_log")

if gaps:
    raise RuntimeError("Silver reconciliation failed for: " + ", ".join(gaps))
print("silver part 2 finished, batch", SILVER_BATCH_ID)

## Check the result
Read-only cells. Run them after the load.

In [ ]:
# Sales by channel and region, in EUR (returns are negative)
(spark.table("silver_sales").groupBy("region", "channel")
      .agg(F.count("*").alias("rows"), F.sum(F.when(F.col("is_return"), 1).otherwise(0)).alias("return_rows"),
           F.round(F.sum("line_amount_eur") / 1e6, 2).alias("net_sales_eur_millions"))
      .orderBy("region", "channel").show(truncate=False))

In [ ]:
# Spot-check the EUR conversion: a weekday sale and a weekend sale, with the rate used
(spark.table("silver_sales").filter(F.col("currency") != "EUR")
      .select("sale_id", "event_date", F.date_format("event_date", "E").alias("weekday"), "currency", "quantity",
              "unit_price", "fx_rate_to_eur", "fx_rate_date", "fx_carried_forward", "line_amount_eur")
      .orderBy(F.col("fx_carried_forward").desc(), "event_date").show(6, truncate=False))
(spark.table("silver_sales").filter(~F.col("fx_carried_forward")).filter(F.col("currency") != "EUR")
      .select("sale_id", "event_date", F.date_format("event_date", "E").alias("weekday"), "currency", "quantity",
              "unit_price", "fx_rate_to_eur", "fx_rate_date", "fx_carried_forward", "line_amount_eur").show(4, truncate=False))

In [ ]:
# What part 2 found in this run
(spark.table("ctl_dq_results").filter(F.col("batch_id") == SILVER_BATCH_ID)
      .select("table_name", "rule", "severity", "rows_failed", "rows_checked").orderBy("table_name", "rule").show(30, truncate=False))

In [ ]:
# Returns: how many were linked to a sale, and how long after the purchase they came back
(spark.table("silver_sales").filter("is_return")
      .agg(F.count("*").alias("returns"), F.count("original_sale_id").alias("linked"),
           F.min("days_to_return").alias("min_days"), F.max("days_to_return").alias("max_days")).show())
(spark.table("silver_sales").filter("is_return and original_sale_id is not null")
      .groupBy("days_to_return").count().orderBy("days_to_return").show(20))

In [ ]:
# Reconciliation: is every cleaned row accounted for?
c = lambda t: spark.table(t).count()
dups = spark.table("ctl_quarantine").filter((F.col("batch_id") == SILVER_BATCH_ID) & (F.col("rule") == "resent_duplicate"))
d = lambda t: dups.filter(F.col("source_table") == t).count()
nofx = spark.table("ctl_quarantine").filter((F.col("batch_id") == SILVER_BATCH_ID) & (F.col("rule") == "no_fx_rate")).count()
rows = [("sales", c("silver_pos_transaction") + c("silver_ecom_order"), c("silver_sales"),
         d("silver_pos_transaction") + d("silver_ecom_order"), nofx),
        ("repairs", c("silver_after_sales_ticket"), c("silver_after_sales"), d("silver_after_sales_ticket"), 0)]
spark.createDataFrame([(a, b, s, du, n, b - s - du - n) for a, b, s, du, n in rows],
                      ["table", "cleaned_rows_in", "kept", "duplicates_removed", "no_fx_rate", "difference"]).show(truncate=False)